In [1]:
import cobra
import pandas as pd
from cobra.io import load_model
from cobra.io import load_json_model, save_json_model, load_matlab_model, save_matlab_model, read_sbml_model, write_sbml_model
from cobra import Model, Reaction, Metabolite, Gene
import numpy as np

In [2]:
model = read_sbml_model('C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\10-19 Research\\11 Data\\11.09 Models\\Manual_curation\\Energy_consumption\\241209_NGAM_coef_update_GAM.sbml')

In [31]:
# Define nitrogen sources (replace with IDs from your model)
nitrogen_sources = ["EX_nh4_e", "EX_no3_e", "EX_no2_e",'EX_urea_e','EX_n2_e']

# Initialize a list to store results
results = []

# Iterate over nitrogen sources
for source in nitrogen_sources:
    with model:
        # Set the uptake rate for the current nitrogen source
        model.reactions.get_by_id(source).lower_bound = -10.0  # Allow uptake
        model.reactions.get_by_id(source).upper_bound = 0.0    # Prevent export

        # Set all other nitrogen sources to 0
        for other_source in nitrogen_sources:
            if other_source != source:
                model.reactions.get_by_id(other_source).lower_bound = 0
                model.reactions.get_by_id(other_source).upper_bound = 0

        # Optimize the model
        solution = model.optimize()
        print(solution.fluxes[source])
        print(solution.objective_value)

        # Store the results
        results.append({
            "Nitrogen Source": source,
            "Growth (h-1)": solution.objective_value,
            "Yield (gDW/mmol substrate)": (solution.objective_value / -model.reactions.get_by_id(source).flux),
            'H2 uptake flux (mmol substrate/gDW/h)': - model.reactions.get_by_id('EX_h2_e').flux,
            'CO2 uptake flux (mmol substrate/gDW/h)': - model.reactions.get_by_id('EX_co2_e').flux,
            'O2 uptake flux (mmol substrate/gDW/h)': - model.reactions.get_by_id('EX_o2_e').flux,
        })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)


# Optional: Save to a CSV for further analysis
results_df.to_csv("nitrogen_source_analysis.csv", index=False)

-0.40413030105220576
0.04619973204490571
-0.3564969841061128
0.04075434358088106
-0.3714380805668502
0.04246239331422905
-0.20893458894942457
0.04777034535272263
-0.18778477494913146
0.04293469834940811
